In [5]:
import pandas as pd, numpy as np
from scipy import stats

cust = pd.read_csv("FinTrust_Customer_Data - FinTrust_Customer_Data.csv")
tx = pd.read_csv("FinTrust_Transaction_Data - FinTrust_Transaction_Data.csv")
tx['flag'] = (tx['Risk_Review_Flag'] == 'Yes').astype(int)
tx['failed'] = (tx['Transaction_Status'] == 'Failed').astype(int)
tx = tx.merge(cust[['Customer_ID', 'Customer_Segment']], on='Customer_ID', how='left')

def chi_p(a, b):
    return stats.chi2_contingency(pd.crosstab(a, b))[1]

# 1. Risk flag by type, channel, segment
print("1. RISK FLAG DRIVERS (chi-square p-values)")
print("   Transaction type:", chi_p(tx['Transaction_Type'], tx['flag']))
print("   Channel:         ", chi_p(tx['Channel'], tx['flag']))
print("   Segment:         ", chi_p(tx['Customer_Segment'], tx['flag']))
print(tx.groupby('Transaction_Type')['flag'].mean().mul(100).round(2).sort_values(ascending=False))

# 2. International vs domestic
print("\n2. INTERNATIONAL vs DOMESTIC")
print(tx.groupby('International_Transaction')[['flag', 'failed']].mean().mul(100).round(2))
print("   Risk p:", chi_p(tx['International_Transaction'], tx['flag']))
print("   Failure p:", chi_p(tx['International_Transaction'], tx['failed']))
print(tx.groupby(['Transaction_Type', 'International_Transaction'])['flag'].mean().mul(100).round(1).unstack())

# 3. Value per customer across segments
print("\n3. VALUE PER CUSTOMER BY SEGMENT")
cv = tx.groupby(['Customer_ID', 'Customer_Segment'])['Amount_NGN'].sum().reset_index()
print(cv.groupby('Customer_Segment')['Amount_NGN'].agg(['mean', 'std', 'count']).round(0))
groups = [g['Amount_NGN'].values for _, g in cv.groupby('Customer_Segment')]
print("   ANOVA p:", stats.f_oneway(*groups)[1])
print("   Kruskal-Wallis p:", stats.kruskal(*groups)[1])

# 4. Are repeat-flagged customers more than chance?
print("\n4. REPEAT-FLAGGED CUSTOMERS vs CHANCE")
tx['p'] = tx.groupby(['Transaction_Type', 'International_Transaction'])['flag'].transform('mean')
observed = (tx.groupby('Customer_ID')['flag'].sum() >= 3).sum()
rng = np.random.default_rng(7)
sims = []
for _ in range(500):
    simulated = (rng.random(len(tx)) < tx['p'].values).astype(int)
    sims.append((pd.Series(simulated).groupby(tx['Customer_ID'].values).sum() >= 3).sum())
print("   Observed:", observed, "| Expected by chance:", round(np.mean(sims), 1),
      "| p:", np.mean(np.array(sims) >= observed))

# 5. Engagement vs activity
print("\n5. ENGAGEMENT vs ACTIVITY")
act = tx.groupby('Customer_ID').agg(n=('Amount_NGN', 'size'), v=('Amount_NGN', 'sum')).reset_index()
act = act.merge(cust[['Customer_ID', 'Digital_Engagement_Score']], on='Customer_ID')
print("   vs count:", stats.pearsonr(act['Digital_Engagement_Score'], act['n']))
print("   vs value:", stats.pearsonr(act['Digital_Engagement_Score'], act['v']))

# 6. Failure rate by channel
print("\n6. FAILURE RATE BY CHANNEL")
print(tx.groupby('Channel')['failed'].mean().mul(100).round(2).sort_values(ascending=False))
print("   All channels p:", chi_p(tx['Channel'], tx['failed']))
print("   Mobile App share of failures:", round(tx[tx['Channel'] == 'Mobile App']['failed'].sum() / tx['failed'].sum(), 3))

# 7. Amount and flagging, ATM high-value
print("\n7. AMOUNT AND FLAGGING")
print("   Mann-Whitney p:", stats.mannwhitneyu(tx[tx.flag == 1]['Amount_NGN'], tx[tx.flag == 0]['Amount_NGN'])[1])
hv = tx[tx['Amount_NGN'] > tx['Amount_NGN'].quantile(0.9)]
print(hv.groupby('Channel')['flag'].agg(['mean', 'count']).round(3))
print("   High-value channel difference p:", chi_p(hv['Channel'], hv['flag']))

1. RISK FLAG DRIVERS (chi-square p-values)
   Transaction type: 2.8826647025229395e-75
   Channel:          0.018657287902798324
   Segment:          0.7514702799624589
Transaction_Type
Transfer           28.49
Cash Withdrawal    25.31
Deposit            16.19
Airtime/Data       15.19
Card Purchase      13.02
Bill Payment       12.81
Name: flag, dtype: float64

2. INTERNATIONAL vs DOMESTIC
                            flag  failed
International_Transaction               
No                         18.88    5.36
Yes                        36.88    2.50
   Risk p: 3.9620396869384127e-22
   Failure p: 0.007986512942087198
International_Transaction    No   Yes
Transaction_Type                     
Airtime/Data               14.4  32.0
Bill Payment               12.3  26.8
Card Purchase              12.1  35.2
Cash Withdrawal            24.9  40.5
Deposit                    15.2  38.6
Transfer                   27.9  42.0

3. VALUE PER CUSTOMER BY SEGMENT
                      mean       std

In [ ]:
from google.colab import drive
drive.mount('/content/drive')